# Paso 1 — Crear la estructura del Data Warehouse
**Archivo original:** `sql/CrearTablas.py`

---

## ¿Qué hace este script?

Crea **todas las tablas vacías** del Data Warehouse en DuckDB.
Es el equivalente a construir los estantes antes de poner los productos.

Pasos internos:
1. Conectarse (o crear) el archivo `aduana.duckdb`
2. Crear el esquema `dw`
3. Borrar tablas anteriores (para empezar limpio)
4. Crear tablas de Staging
5. Crear 12 tablas de Dimensiones
6. Crear la Fact Table


---

## Bloque 1: Importaciones y configuración


In [ ]:
import duckdb  # librería para manejar la base de datos DuckDB
import os      # para operaciones del sistema de archivos

# Ruta al archivo de la base de datos
# DuckDB guarda TODA la base de datos en un único archivo .duckdb
DB_PATH = r"C:\Información\proyectos\aduana_bi\db\aduana.duckdb"

# Crear la carpeta 'db' si no existe (evita error de ruta no encontrada)
os.makedirs(os.path.dirname(DB_PATH), exist_ok=True)

print(f"Ruta de la base de datos: {DB_PATH}")

---

## Bloque 2: Conexión a DuckDB

`duckdb.connect(ruta)` crea el archivo si no existe, o lo abre si ya existe.
A diferencia de MySQL o PostgreSQL, **no hay servidor**: todo está en el archivo.


In [ ]:
con = duckdb.connect(DB_PATH)
print("Conexión establecida correctamente.")
print("Creando estructura DW...")

---

## Bloque 3: Crear el Esquema

Un **esquema** (SCHEMA) es como una carpeta dentro de la base de datos.
Agrupa las tablas relacionadas bajo un mismo nombre.

Aquí usamos el esquema `dw` (Data Warehouse) para separar nuestras tablas
de otras posibles tablas que pudiera tener la misma base de datos.


In [ ]:
# IF NOT EXISTS: si el esquema ya existe, no lanza error, lo ignora
con.execute("CREATE SCHEMA IF NOT EXISTS dw;")
print("Esquema 'dw' creado (o ya existía).")

---

## Bloque 4: Limpieza — Borrar tablas anteriores

Se borran las tablas en orden inverso al de dependencias:
- Primero la **Fact Table** (depende de las dimensiones)
- Luego las **Dimensiones**
- Finalmente las tablas de **Staging**

Esto permite ejecutar el script múltiples veces sin errores.


In [ ]:
# DROP TABLE IF EXISTS: borra la tabla si existe, no hace nada si no existe
tablas_a_borrar = [
    "dw.fact_aduana_item",
    "dw.dim_operacion", "dw.dim_destinacion", "dw.dim_regimen",
    "dw.dim_aduana", "dw.dim_pais", "dw.dim_producto",
    "dw.dim_medio_transporte", "dw.dim_canal", "dw.dim_unidad_medida",
    "dw.dim_acuerdo", "dw.dim_marca", "dw.dim_fecha",
    "dw.stg_destinaciones", "dw.stg_aduana"
]

for tabla in tablas_a_borrar:
    con.execute(f"DROP TABLE IF EXISTS {tabla};")
    print(f"  Tabla {tabla} eliminada (o no existía).")

---

## Bloque 5: Crear tablas de Staging

Las tablas de **Staging** son tablas temporales de paso.
Reciben los datos del Excel tal cual (solo normalizados), antes de transformarlos
en dimensiones y hechos.

### `stg_aduana` — Tabla principal de staging
41 columnas que replican exactamente las columnas del Excel de despachos.
Tipos de datos ya definidos: VARCHAR para texto, DATE para fechas, DOUBLE para números.


In [ ]:
con.execute("""
CREATE TABLE dw.stg_aduana (
    despacho_cifrado VARCHAR,           -- Código único del despacho (ej: '2024-000123')
    operacion VARCHAR,                  -- 'IMPORTACION' o 'EXPORTACION'
    destinacion VARCHAR,                -- Código del régimen aduanero (ej: 'IM4')
    regimen VARCHAR,                    -- Descripción del régimen
    oficializacion DATE,                -- Fecha de oficialización del despacho
    cancelacion DATE,                   -- Fecha de cancelación/cierre
    anio INTEGER,                       -- Año extraído de la fecha
    mes VARCHAR,                        -- Nombre del mes
    aduana VARCHAR,                     -- Aduana donde se procesó
    cotizacion DOUBLE,                  -- Tipo de cambio aplicado
    medio_transporte VARCHAR,           -- Transporte usado (TERRESTRE, AEREO, etc.)
    canal VARCHAR,                      -- Canal de control (ROJO, VERDE, AMARILLO)
    item INTEGER,                       -- Número de línea dentro del despacho
    pais_origen VARCHAR,                -- País de origen (formato: 'COD - DESCRIPCION')
    pais_procedencia_destino VARCHAR,   -- País procedencia/destino (mismo formato)
    uso VARCHAR,                        -- Destino final del bien
    unidad_medida_estadistica VARCHAR,  -- Unidad de medida estadística
    cantidad_estadistica DOUBLE,        -- Cantidad en unidad estadística
    kilo_neto DOUBLE,                   -- Peso neto en kilogramos
    kilo_bruto DOUBLE,                  -- Peso bruto en kilogramos
    fob_dolar DOUBLE,                   -- Valor FOB en dólares
    flete_dolar DOUBLE,                 -- Flete en dólares
    seguro_dolar DOUBLE,                -- Seguro en dólares
    imponible_dolar DOUBLE,             -- Base imponible en dólares
    imponible_gs DOUBLE,                -- Base imponible en guaraníes
    ajuste_a_incluir DOUBLE,            -- Ajuste de valor a incluir
    ajuste_a_deducir DOUBLE,            -- Ajuste de valor a deducir
    posicion VARCHAR,                   -- Posición NCM (Nomenclatura Común del Mercosur)
    rubro VARCHAR,                      -- Rubro/categoría arancelaria
    desc_capitulo VARCHAR,              -- Descripción del capítulo arancelario
    desc_posicion VARCHAR,              -- Descripción de la posición NCM
    desc_partida VARCHAR,               -- Descripción de la partida
    mercaderia VARCHAR,                 -- Descripción de la mercadería
    marca_item VARCHAR,                 -- Marca del producto
    acuerdo VARCHAR,                    -- Acuerdo comercial aplicado
    derecho DOUBLE,                     -- Arancel de importación
    isc DOUBLE,                         -- Impuesto Selectivo al Consumo
    servicio DOUBLE,                    -- Tasas de servicio
    renta DOUBLE,                       -- Impuesto a la Renta
    iva DOUBLE,                         -- IVA aplicado
    otros DOUBLE,                       -- Otros impuestos
    total DOUBLE                        -- Total de tributos
);
""")
print("stg_aduana creada.")

In [ ]:
# stg_destinaciones: catálogo auxiliar de códigos de destinación aduanera
# Viene de un Excel separado (LISTADO_DE_DESTINACIONES.xlsx)
con.execute("""
CREATE TABLE dw.stg_destinaciones (
    cod_destinacion VARCHAR,        -- Código corto (ej: 'IM4')
    descripcion_dest VARCHAR,       -- Nombre completo del régimen
    tipo_regimen_base VARCHAR,      -- SUSPENSIVO / DEFINITIVO / TEMPORAL
    tipo_operacion_base VARCHAR     -- IMPORT / EXPORT
);
""")
print("stg_destinaciones creada.")

---

## Bloque 6: Crear las 12 Tablas de Dimensiones

Cada dimensión tiene:
- Una **clave surrogate** (`id_xxx INTEGER PRIMARY KEY`) — generada por nosotros, secuencial
- Una **clave natural** (el valor real del negocio) — declarada como `UNIQUE`
- Atributos descriptivos adicionales


In [ ]:
# dim_operacion: solo 2 valores posibles: IMPORTACION, EXPORTACION
# Los flags booleanos facilitan filtrar en Power BI
con.execute("""
CREATE TABLE dw.dim_operacion (
    id_operacion INTEGER PRIMARY KEY,
    operacion VARCHAR UNIQUE,
    es_importacion BOOLEAN,
    es_exportacion BOOLEAN
);
""")

# dim_destinacion: régimen aduanero específico (ej: IM4 = Importación definitiva)
# Se enriquece con el catálogo stg_destinaciones
con.execute("""
CREATE TABLE dw.dim_destinacion (
    id_destinacion INTEGER PRIMARY KEY,
    cod_destinacion VARCHAR UNIQUE,
    descripcion_dest VARCHAR,
    tipo_regimen_base VARCHAR,
    tipo_operacion_base VARCHAR
);
""")

# dim_regimen: tipo general de régimen (más general que destinacion)
con.execute("""
CREATE TABLE dw.dim_regimen (
    id_regimen INTEGER PRIMARY KEY,
    regimen VARCHAR UNIQUE
);
""")

# dim_aduana: aduanas físicas del país donde se procesó el despacho
con.execute("""
CREATE TABLE dw.dim_aduana (
    id_aduana INTEGER PRIMARY KEY,
    aduana VARCHAR UNIQUE
);
""")

# dim_pais: tabla unificada de países (origen + destino usan la misma dimensión)
# El campo codigo_pais viene de separar 'ARG - ARGENTINA' en código y descripción
con.execute("""
CREATE TABLE dw.dim_pais (
    id_pais INTEGER PRIMARY KEY,
    codigo_pais VARCHAR UNIQUE,
    descripcion_pais VARCHAR
);
""")

# dim_producto: clasificación arancelaria NCM + descripción de la mercadería
# La clave es COMPUESTA (combinación de 6 campos) porque el mismo código NCM
# puede tener distintas descripciones según la subpartida
con.execute("""
CREATE TABLE dw.dim_producto (
    id_producto INTEGER PRIMARY KEY,
    posicion_ncm VARCHAR,
    rubro VARCHAR,
    desc_capitulo VARCHAR,
    desc_posicion VARCHAR,
    desc_partida VARCHAR,
    mercaderia VARCHAR
);
""")

print("Primeras 6 dimensiones creadas.")

In [ ]:
# dim_medio_transporte: TERRESTRE, AEREO, MARITIMO, etc.
con.execute("""
CREATE TABLE dw.dim_medio_transporte (
    id_medio_transporte INTEGER PRIMARY KEY,
    medio_transporte VARCHAR UNIQUE
);
""")

# dim_canal: canal de control aduanero (ROJO=inspección física, VERDE=sin inspección)
con.execute("""
CREATE TABLE dw.dim_canal (
    id_canal INTEGER PRIMARY KEY,
    canal VARCHAR UNIQUE
);
""")

# dim_unidad_medida: unidades estadísticas (KG, LT, UN, M2, etc.)
con.execute("""
CREATE TABLE dw.dim_unidad_medida (
    id_unidad_medida INTEGER PRIMARY KEY,
    unidad_medida VARCHAR UNIQUE
);
""")

# dim_acuerdo: acuerdos comerciales aplicados (MERCOSUR, ACE, etc.)
con.execute("""
CREATE TABLE dw.dim_acuerdo (
    id_acuerdo INTEGER PRIMARY KEY,
    acuerdo VARCHAR UNIQUE
);
""")

# dim_marca: marca del producto (puede estar vacía en muchos registros)
con.execute("""
CREATE TABLE dw.dim_marca (
    id_marca INTEGER PRIMARY KEY,
    marca VARCHAR UNIQUE
);
""")

# dim_fecha: dimensión de tiempo con atributos derivados
# Se usa DOS VECES en la fact table: una para oficializacion y otra para cancelacion
con.execute("""
CREATE TABLE dw.dim_fecha (
    id_fecha INTEGER PRIMARY KEY,
    fecha DATE UNIQUE,
    anio INTEGER,
    mes_numero INTEGER,
    mes_nombre VARCHAR,
    trimestre INTEGER,
    anio_mes VARCHAR          -- formato 'YYYY-MM' para ordenar cronológicamente
);
""")

print("Últimas 6 dimensiones creadas.")

---

## Bloque 7: Crear la Fact Table

La **Fact Table** es el corazón del modelo. Contiene:
- Una **clave primaria** propia (`id_fact BIGINT`)
- La **clave natural** del negocio (`despacho_cifrado + item`)
- **15 claves foráneas** apuntando a cada dimensión
- **19 métricas numéricas** (pesos, valores, impuestos)
- `uso`: atributo descriptivo que no justifica una dimensión propia


In [ ]:
con.execute("""
CREATE TABLE dw.fact_aduana_item (
    -- Clave primaria surrogate
    id_fact BIGINT PRIMARY KEY,

    -- Clave natural del negocio (identifica unívocamente cada línea)
    despacho_cifrado VARCHAR,
    item INTEGER,

    -- Claves foráneas hacia las 14 dimensiones
    id_operacion INTEGER,
    id_destinacion INTEGER,
    id_regimen INTEGER,
    id_aduana INTEGER,
    id_pais_origen INTEGER,           -- misma dim_pais, rol 'origen'
    id_pais_destino INTEGER,          -- misma dim_pais, rol 'destino'
    id_producto INTEGER,
    id_medio_transporte INTEGER,
    id_canal INTEGER,
    id_unidad_medida INTEGER,
    id_acuerdo INTEGER,
    id_marca INTEGER,
    id_fecha_oficializacion INTEGER,  -- misma dim_fecha, rol 'oficializacion'
    id_fecha_cancelacion INTEGER,     -- misma dim_fecha, rol 'cancelacion'

    -- Atributo de baja cardinalidad que no justifica dimensión propia
    uso VARCHAR,

    -- Métricas numéricas (hechos aditivos)
    cantidad_estadistica DOUBLE,
    kilo_neto DOUBLE,
    kilo_bruto DOUBLE,
    fob_dolar DOUBLE,
    flete_dolar DOUBLE,
    seguro_dolar DOUBLE,
    imponible_dolar DOUBLE,
    imponible_gs DOUBLE,
    ajuste_a_incluir DOUBLE,
    ajuste_a_deducir DOUBLE,
    derecho DOUBLE,
    isc DOUBLE,
    servicio DOUBLE,
    renta DOUBLE,
    iva DOUBLE,
    otros DOUBLE,
    total DOUBLE,

    -- Constraint: la combinación despacho+item debe ser única
    UNIQUE(despacho_cifrado, item)
);
""")
print("fact_aduana_item creada.")

---

## Bloque 8: Cerrar la conexión y verificar


In [ ]:
con.close()
print("DW creado correctamente.")

In [ ]:
# Verificación: abrir de nuevo y listar las tablas creadas
con = duckdb.connect(DB_PATH)
tablas = con.execute("""
    SELECT table_name, table_type
    FROM information_schema.tables
    WHERE table_schema = 'dw'
    ORDER BY table_name
""").fetchall()

print(f"Tablas en el esquema 'dw': {len(tablas)}\n")
for tabla in tablas:
    print(f"  {tabla[0]}")

con.close()

---

**Resultado esperado:** 15 tablas (2 staging + 12 dimensiones + 1 fact), todas con 0 filas.

**Siguiente paso:** `02_Cargar_Staging.ipynb`
